# 4 — One risk number for a mixed equity and options book

The three previous notebooks measured different things separately. This one is
the integration: a book holding **both shares and options**, revalued against
the **same** simulated scenarios, producing a single VaR.

This is the piece that separates an engine from a folder of scripts. An option
pricer that cannot feed the VaR model leaves the derivatives risk uncounted.

In [1]:
from risk_engine import load_prices, to_returns, Portfolio, historical_var, parametric_var

TICKERS = ["AAPL", "MSFT"]
prices = load_prices(TICKERS, start="2018-01-01")
returns = to_returns(prices)
spot = {t: float(prices[t].iloc[-1]) for t in TICKERS}
spot

{'AAPL': 313.3299865722656, 'MSFT': 499.989990234375}

In [2]:
book = Portfolio()
book.add_equity("AAPL", quantity=1000, price=spot["AAPL"])
book.add_equity("MSFT", quantity=400,  price=spot["MSFT"])

# A protective put on the AAPL position, 5% out of the money.
book.add_option(
    underlying="AAPL", kind="put",
    strike=spot["AAPL"] * 0.95, expiry_years=0.25,
    volatility=0.28, quantity=10, spot=spot["AAPL"], rate=0.04,
)

print(book.summary())

Market value      : 522,575.12
Equity positions  : 2
Option positions  : 1
Risk factors      : AAPL, MSFT


## Exposure is not market value

An option's market value is its premium. Its *risk* is the equity position it
behaves like — `delta x contract_size x quantity x spot`. That delta-equivalent
number is the correct denominator for a concentration limit; option notional is
not.

In [3]:
book.exposure_by_underlying().round(2)

AAPL    217515.69
MSFT    199996.00
Name: delta_equivalent_exposure, dtype: float64

In [4]:
book.greeks_by_underlying().round(4)

,delta,gamma,vega,theta,rho
underlying,,,,,
AAPL,694.2064,7.9944,54939.6921,-26563.6903,-26265.8578
MSFT,400.0000,0.0000,0.0000,0.0000,0.0000


Read the AAPL row: the long put contributes **negative delta** (it offsets the
shares), **positive gamma** (the hedge strengthens as the market falls), and
**negative theta** (that protection decays daily). Those three numbers are the
entire economics of a protective put.

## Revaluing the whole book

Each historical day is treated as a scenario. Equity legs are exact. Option legs
use a delta-gamma expansion:

$$\Delta V \approx \delta \cdot \Delta S + \tfrac{1}{2}\gamma \cdot (\Delta S)^2$$

Full repricing on every path would be more accurate but quadratically more
expensive, and the gamma term already captures the convexity that makes a
delta-only VaR wrong for an option book. The approximation degrades for very
large moves and near expiry — stated because the limitation is real.

In [5]:
pnl_returns = book.scenario_returns(returns)

for conf in (0.95, 0.99):
    h = historical_var(pnl_returns, conf)
    print(f"{conf:.0%} VaR: {h.var:.4%}    ES: {h.expected_shortfall:.4%}")

95% VaR: 2.0214%    ES: 2.9446%
99% VaR: 3.3733%    ES: 4.4119%


## Does the hedge actually work?

The test that matters: build the same book **without** the put and compare the
tail. A protective put should cut the downside while costing a little in
premium.

In [6]:
import pandas as pd

naked = Portfolio()
naked.add_equity("AAPL", quantity=1000, price=spot["AAPL"])
naked.add_equity("MSFT", quantity=400,  price=spot["MSFT"])

rows = []
for label, p in [("unhedged", naked), ("with protective put", book)]:
    rets = p.scenario_returns(returns)
    v99 = historical_var(rets, 0.99)
    rows.append({
        "book": label,
        "market value": p.market_value,
        "99% VaR": v99.var,
        "99% ES": v99.expected_shortfall,
        "worst day": rets.min(),
    })

pd.DataFrame(rows).style.format({
    "market value": "{:,.0f}", "99% VaR": "{:.4%}",
    "99% ES": "{:.4%}", "worst day": "{:.4%}",
})

,book,market value,99% VaR,99% ES,worst day
0,unhedged,"513,326",4.4582%,6.0641%,-13.5949%
1,with protective put,"522,575",3.3733%,4.4119%,-9.7528%


The hedged book shows a smaller tail loss — and the improvement is larger in
**ES** than in VaR, because the put pays off most in exactly the extreme
scenarios that ES averages over and VaR ignores.

In [7]:
# Convexity, directly: a stress ladder across parallel moves in both names.
stress = pd.DataFrame(
    {"AAPL": [-0.30, -0.20, -0.10, -0.05, 0.0, 0.05, 0.10],
     "MSFT": [-0.30, -0.20, -0.10, -0.05, 0.0, 0.05, 0.10]}
)
ladder = pd.DataFrame({
    "shock": stress["AAPL"].map("{:.0%}".format),
    "unhedged P&L": naked.scenario_pnl(stress),
    "hedged P&L": book.scenario_pnl(stress),
})
ladder["hedge benefit"] = ladder["hedged P&L"] - ladder["unhedged P&L"]
ladder.style.format({"unhedged P&L": "{:,.0f}", "hedged P&L": "{:,.0f}",
                     "hedge benefit": "{:,.0f}"})

,shock,unhedged P&L,hedged P&L,hedge benefit
0,-30%,"-153,998","-89,935","64,063"
1,-20%,"-102,665","-67,805","34,860"
2,-10%,"-51,333","-37,827","13,506"
3,-5%,"-25,666","-19,895","5,772"
4,0%,0,0,0
5,5%,"25,666","21,857","-3,810"
6,10%,"51,333","45,675","-5,657"


The hedge benefit grows non-linearly as the shock deepens — that is gamma. A
delta-only model would show a constant benefit and materially understate the
protection.

**Caveat worth stating:** at a -30% shock the delta-gamma expansion is being
pushed past where it is reliable. Full repricing is the right tool for stress
scenarios of that size; the expansion is appropriate for the daily VaR
distribution, where moves are small.